# 🏭 Vision-Driven Industrial Safety & Quality Inspection Engine
## Notebook 02 — Improve Existing 7-Class Textile Model

**Prerequisite:** Notebook 01 completed OR only textile dataset available.

---

### Purpose
Run Experiment B: a controlled improvement of the original YOLOv8s textile-defect model.

### Historical Baseline (Experiment A) — DO NOT MODIFY
| Metric | Value |
|---|---|
| Precision | 76.97% |
| Recall | 74.76% |
| mAP@50 | 78.82% |
| mAP@50-95 | 47.12% |

### Per-class baseline (reported values)
| Class | mAP@50 | Status |
|---|---|---|
| contamination | 99.0% | Strong |
| stain | 91.9% | Strong |
| baekra | 83.9% | Needs improvement |
| cut | 82.8% | Needs improvement |
| gray stitch | 74.0% | Needs improvement |
| selvet | 68.9% | Needs improvement |
| color issues | 51.2% | Weakest — needs improvement |

### Improvement strategy (justified by Notebook 02 diagnosis)
- Transfer learn from `best.pt` (same 7 classes, no head change)
- Longer training: 50 epochs with early stopping
- Improved augmentation targeting weak classes
- Better LR schedule (cosine)
- Scale augmentation (helps smaller objects: gray stitch, selvet)


In [ ]:
%pip install -q ultralytics PyYAML matplotlib seaborn


---


In [ ]:
import torch, subprocess
from pathlib import Path
import ultralytics

print('=== GPU CHECK ===')
assert torch.cuda.is_available(), 'No GPU! Use Colab T4 kernel.'
props = torch.cuda.get_device_properties(0)
print(f'GPU         : {props.name}')
print(f'VRAM        : {props.total_memory/1e9:.1f} GB')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')

# nvidia-smi summary
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,utilization.gpu',
                        '--format=csv,noheader'], capture_output=True, text=True, timeout=10)
    if r.returncode == 0:
        print(f'nvidia-smi  : {r.stdout.strip()}')
except Exception:
    pass



---


In [ ]:
from pathlib import Path
import shutil, json

CONTENT = Path('/content')
RUNS_DIR = CONTENT / 'runs'

# ── Locate textile dataset ───────────────────────────────────────────────────
DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    DRIVE_PROJECT = DRIVE_ROOT / 'Hangzhou_Textile_POC'
except ImportError:
    DRIVE_PROJECT = None

# Textile dataset
TEXTILE_DATASET = None
if DRIVE_PROJECT and (DRIVE_PROJECT / 'final_dataset').exists():
    TEXTILE_DATASET = DRIVE_PROJECT / 'final_dataset'
elif (CONTENT / 'textile_extracted' / 'final_dataset').exists():
    TEXTILE_DATASET = CONTENT / 'textile_extracted' / 'final_dataset'

if TEXTILE_DATASET is None:
    # Try extracting ZIP
    zips = list(CONTENT.glob('textile_defect_yolov8_final.zip'))
    if DRIVE_PROJECT:
        zips += list(DRIVE_PROJECT.glob('textile_defect_yolov8_final.zip'))
    if zips:
        import zipfile
        print(f'Extracting: {zips[0]}')
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(CONTENT / 'textile_extracted')
        TEXTILE_DATASET = CONTENT / 'textile_extracted' / 'final_dataset'
    if not TEXTILE_DATASET or not TEXTILE_DATASET.exists():
        raise FileNotFoundError(
            'Textile dataset not found. Upload textile_defect_yolov8_final.zip '
            'or ensure Drive is mounted with the dataset in Hangzhou_Textile_POC/final_dataset/'
        )

print(f'Textile dataset: {TEXTILE_DATASET}')

# ── Locate baseline checkpoint ────────────────────────────────────────────────
BASELINE_PT = None
candidates = []
if DRIVE_PROJECT:
    candidates += [
        DRIVE_PROJECT / 'Training_Results' / 'yolov8s_textile_initial' / 'best.pt',
        DRIVE_PROJECT / 'best.pt',
    ]
candidates += [CONTENT / 'best.pt']

for c in candidates:
    if c.exists():
        BASELINE_PT = c
        break

if BASELINE_PT:
    print(f'Baseline checkpoint: {BASELINE_PT}')
else:
    print('⚠ baseline best.pt not found on Drive — will use YOLOv8s pretrained weights instead.')
    print('  For best results, place the baseline checkpoint in Training_Results/yolov8s_textile_initial/best.pt')
    if DRIVE_PROJECT:
        print(f'  {DRIVE_PROJECT}/best.pt')
    BASELINE_PT = 'yolov8s.pt'  # fall back to ImageNet pretrained

# Output dir
IMPROVED_RUNS = CONTENT / 'runs' / 'improved_7class'


In [ ]:
"""Ensure data.yaml has an absolute path (required for Colab training)."""
import yaml

yaml_path = TEXTILE_DATASET / 'data.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

# Update path to absolute Colab path
cfg['path'] = str(TEXTILE_DATASET)
cfg['train'] = 'images/train'
cfg['val'] = 'images/val'
cfg['test'] = 'images/test'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(f'data.yaml updated: path={cfg["path"]}')


---


In [ ]:
from ultralytics import YOLO
import time

print('=== EXPERIMENT B — IMPROVED 7-CLASS TRAINING ===')
print(f'Starting checkpoint : {BASELINE_PT}')
print(f'Dataset             : {yaml_path}')

model = YOLO(str(BASELINE_PT) if str(BASELINE_PT) != 'yolov8s.pt' else 'yolov8s.pt')

t0 = time.time()

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=-1,           # auto-batch for T4
    device=0,           # GPU 0
    project=str(CONTENT / 'runs'),
    name='improved_7class',
    exist_ok=True,

    # Optimizer
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,           # final LR = lr0 * lrf → cosine decay to 0.0001
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Early stopping
    patience=10,

    # Augmentation — carefully tuned for textile defects
    hsv_h=0.015,        # subtle hue shift (helps color issues)
    hsv_s=0.7,          # saturation (helps color issues)
    hsv_v=0.4,          # brightness variation
    degrees=10.0,       # rotation ±10° (same as existing pipeline)
    translate=0.1,
    scale=0.5,          # scale ±50% (helps detect smaller gray stitch / selvet)
    shear=0.0,          # no shear (could distort textile patterns)
    perspective=0.0,    # no perspective warp
    flipud=0.0,         # NO vertical flip (textile orientation is meaningful)
    fliplr=0.5,         # horizontal flip OK
    mosaic=1.0,         # mosaic augmentation
    mixup=0.1,          # mild mixup (helps underrepresented classes)
    copy_paste=0.0,     # no copy-paste (could create invalid textile patterns)

    # Mixed precision
    amp=True,

    # Logging
    verbose=True,
    plots=True,
)

elapsed = (time.time() - t0) / 60
print(f'\nTraining completed in {elapsed:.1f} minutes')

# Locate best checkpoint
IMPROVED_BEST_PT = CONTENT / 'runs' / 'improved_7class' / 'weights' / 'best.pt'
print(f'Best checkpoint: {IMPROVED_BEST_PT}')


---


In [ ]:
from ultralytics import YOLO
import json

print('=== EVALUATING IMPROVED 7-CLASS MODEL ===')

improved_model = YOLO(str(IMPROVED_BEST_PT))
val_results = improved_model.val(
    data=str(yaml_path),
    split='val',
    device=0,
    imgsz=640,
    batch=16,
    verbose=True,
    plots=True,
    project=str(CONTENT / 'runs'),
    name='improved_7class_eval',
    exist_ok=True,
)

# Extract metrics
box = val_results.box
overall_metrics = {
    'precision': float(box.mp),        # mean precision
    'recall': float(box.mr),           # mean recall
    'mAP50': float(box.map50),
    'mAP50_95': float(box.map),
}

# Per-class metrics
class_names = improved_model.names
per_class = {}
if hasattr(box, 'ap_class_index'):
    for i, cls_idx in enumerate(box.ap_class_index):
        name = class_names[int(cls_idx)]
        per_class[name] = {
            'precision': float(box.p[i]),
            'recall': float(box.r[i]),
            'ap50': float(box.ap50[i]),
            'ap50_95': float(box.ap[i]),
        }

print('\n=== EXPERIMENT B RESULTS ===')
print(f'Precision   : {overall_metrics["precision"]*100:.2f}%')
print(f'Recall      : {overall_metrics["recall"]*100:.2f}%')
print(f'mAP@50      : {overall_metrics["mAP50"]*100:.2f}%')
print(f'mAP@50-95   : {overall_metrics["mAP50_95"]*100:.2f}%')

print(f'\n{"Class":<20} {"P%":>7} {"R%":>7} {"AP@50%":>9} {"AP@50-95%":>11}')
print('-' * 60)

HISTORICAL_MAP50 = {
    'baekra': 0.839, 'color issues': 0.512, 'contamination': 0.990,
    'cut': 0.828, 'gray stitch': 0.740, 'selvet': 0.689, 'stain': 0.919
}

for cls_name in sorted(per_class.keys()):
    m = per_class[cls_name]
    hist = HISTORICAL_MAP50.get(cls_name, 0)
    delta = m['ap50'] - hist
    direction = f'▲ +{delta*100:.1f}%' if delta >= 0 else f'▼ {delta*100:.1f}%'
    print(f'{cls_name:<20} {m["precision"]*100:>7.1f} {m["recall"]*100:>7.1f} '
          f'{m["ap50"]*100:>9.1f} {m["ap50_95"]*100:>11.1f}  {direction}')

# Save results
eval_results_path = CONTENT / 'runs' / 'improved_7class_eval' / 'results.json'
eval_results_path.parent.mkdir(parents=True, exist_ok=True)
experiment_b = {
    'experiment': 'B - Improved 7-class',
    'checkpoint': str(IMPROVED_BEST_PT),
    'overall': overall_metrics,
    'per_class': per_class,
    'historical_baseline': {
        'precision': 0.7697,
        'recall': 0.7476,
        'mAP50': 0.7882,
        'mAP50_95': 0.4712
    }
}
with open(eval_results_path, 'w') as f:
    json.dump(experiment_b, f, indent=2)


---


In [ ]:
"""The confusion matrix is auto-saved by val() to the eval project dir.
Display it here for review."""
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

eval_dir = CONTENT / 'runs' / 'improved_7class_eval'

# Find confusion matrix
cm_files = list(eval_dir.glob('confusion_matrix*.png'))
if cm_files:
    cm_img = Image.open(cm_files[0])
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(cm_img)
    ax.axis('off')
    ax.set_title('Improved 7-Class Model — Confusion Matrix (Normalized)', fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f'Confusion matrix: {cm_files[0]}')
    
    # Copy to reports
    import shutil
    REPORTS_DIR = CONTENT / 'hazard_reports'
    REPORTS_DIR.mkdir(exist_ok=True)
    shutil.copy(cm_files[0], REPORTS_DIR / 'improved_7class_confusion_matrix.png')
else:
    print('Confusion matrix file not found in:', eval_dir)


---


In [ ]:
import shutil, datetime

if DRIVE_PROJECT:
    drive_improved = DRIVE_PROJECT / 'Training_Results' / 'yolov8s_improved_7class'
    drive_improved.mkdir(parents=True, exist_ok=True)
    
    # Copy best.pt
    shutil.copy(IMPROVED_BEST_PT, drive_improved / 'best.pt')
    
    # Copy eval results
    eval_dir = CONTENT / 'runs' / 'improved_7class_eval'
    for f in eval_dir.glob('*.png'):
        shutil.copy(f, drive_improved / f.name)
    for f in eval_dir.glob('*.csv'):
        shutil.copy(f, drive_improved / f.name)
    shutil.copy(eval_results_path, drive_improved / 'results.json')
    
    # Also copy training plots
    training_dir = CONTENT / 'runs' / 'improved_7class'
    for f in training_dir.glob('results.png'):
        shutil.copy(f, drive_improved / 'training_results.png')
    
    print(f'Improved model saved to Drive: {drive_improved}')
    print('Contents:', [f.name for f in drive_improved.iterdir()])
else:
    print('Drive not mounted — checkpoint remains at:')
    print(f'  {IMPROVED_BEST_PT}')
    print('Mount Drive and re-run this cell to save permanently.')



---
## Step 7 — (Optional) Second Iteration



In [ ]:
"""Second iteration analysis — run only if Experiment B underperforms.

Check which classes still underperform and consider:
  - Longer training (75 epochs)
  - Higher scale augmentation for tiny-object classes
  - Class weights via oversampling
  - Adam optimizer if SGD is not converging
"""

# Analyze per-class results from experiment_b
TARGET_MAP50 = 0.80  # target: all classes above 80%
still_weak = {}
for cls_name, metrics in per_class.items():
    if metrics['ap50'] < TARGET_MAP50:
        hist = HISTORICAL_MAP50.get(cls_name, 0)
        improvement = metrics['ap50'] - hist
        still_weak[cls_name] = {
            'ap50': metrics['ap50'],
            'historical': hist,
            'improvement': improvement
        }

if not still_weak:
    print('All classes above target threshold — no second iteration needed.')
else:
    print(f'Classes still below {TARGET_MAP50*100:.0f}% mAP@50:')
    for cls_name, info in sorted(still_weak.items(), key=lambda x: x[1]['ap50']):
        print(f'  {cls_name:<20}: {info["ap50"]*100:.1f}% '
              f'(hist={info["historical"]*100:.1f}%, delta={info["improvement"]*100:+.1f}%)')
    
    print('\nRecommendation: Run second iteration with Adam optimizer and longer training.')
    print('Uncomment below to run second iteration if resources allow.')

# ─────────────────────────────────────────────────────────────────────────────
# UNCOMMENT BELOW TO RUN SECOND ITERATION:
# model2 = YOLO(str(IMPROVED_BEST_PT))
# results2 = model2.train(
#     data=str(yaml_path),
#     epochs=75,
#     imgsz=640,
#     batch=-1,
#     device=0,
#     project=str(CONTENT / 'runs'),
#     name='improved_7class_v2',
#     exist_ok=True,
#     optimizer='Adam',
#     lr0=0.001,
#     lrf=0.01,
#     patience=15,
#     scale=0.7,          # higher scale for tiny objects
#     hsv_s=0.9,          # more aggressive saturation
#     flipud=0.0,
#     fliplr=0.5,
#     mosaic=1.0,
#     mixup=0.15,
#     amp=True,
# )


---
## End of Notebook 02

**Next:** Open `03_train_expanded_model.ipynb`

**Before proceeding, record:**
- [ ] Improved model mAP@50 vs historical baseline
- [ ] Per-class improvements for: baekra, cut, gray stitch, selvet, color issues
- [ ] Regressions (if any) on: contamination, stain
- [ ] Confusion matrix reviewed
